In [1]:
using LinearAlgebra
using Printf
using Statistics
using Plots
using Dates
using ProgressMeter

# --- 0. 保存先の準備 ---
desktop_path = joinpath(homedir(), "Desktop", "GraphResults_Hubbard_CRM_HF_2D")
mkpath(desktop_path)
println("Images will be saved to: $desktop_path")

# ==========================================
# 1. ハミルトニアンと演算子の構築関数 (汎用化)
# ==========================================
function make_matrix(ops...)
    r = ones(1, 1)
    for op in reverse(ops)
        r = kron(op, r)
    end
    return r
end

# サイト数 N に応じて、必要な次元の生成演算子を自動で組み立てる
function get_creation_ops(N_sites=6)
    cdag = [0.0 0.0; 1.0 0.0]
    I2   = [1.0 0.0; 0.0 1.0]
    F_op = [1.0 0.0; 0.0 -1.0]
    Cdag = Dict{String, Matrix{Float64}}()
    N_qubits = 2 * N_sites
    
    for i in 1:N_sites
        for (s_idx, s_name) in enumerate(["u", "d"])
            q_idx = 2 * (i - 1) + s_idx # 1 から 12 までのインデックス
            ops = Matrix{Float64}[]
            for j in 1:N_qubits
                if j < q_idx
                    push!(ops, I2)
                elseif j == q_idx
                    push!(ops, cdag)
                else
                    push!(ops, F_op)
                end
            end
            Cdag["$(i)$(s_name)"] = make_matrix(ops...)
        end
    end
    return Cdag
end

function build_hamiltonian(t_x, t_y, U, mu, N_sites=6)
    dim = 2^(2 * N_sites)
    Cdag = get_creation_ops(N_sites)
    C_op = Dict(k => v' for (k, v) in Cdag)
    N_op = Dict(k => Cdag[k] * C_op[k] for k in keys(Cdag))
    hamil = zeros(Float64, dim, dim)
    
    # 3x2 のエッジ定義
    # 1 - 2 - 3
    # |   |   |
    # 4 - 5 - 6
    edges = [(1, 2, t_x), (2, 3, t_x), (4, 5, t_x), (5, 6, t_x), # 水平 (x方向)
             (1, 4, t_y), (2, 5, t_y), (3, 6, t_y)]              # 垂直 (y方向)
             
    for (i, j, t) in edges
        for s in ["u", "d"]
            k1, k2 = "$(i)$s", "$(j)$s"
            hamil .+= -t .* (Cdag[k1] * C_op[k2] .+ Cdag[k2] * C_op[k1])
        end
    end
    
    for i in 1:N_sites
        hamil .+= U .* (N_op["$(i)u"] * N_op["$(i)d"])
        hamil .+= -mu .* N_op["$(i)u"]
        hamil .+= -mu .* N_op["$(i)d"]
    end
    return hamil
end

function get_particle_number_ops(N_sites=6)
    dim = 2^(2 * N_sites)
    Cdag = get_creation_ops(N_sites)
    C_op = Dict(k => v' for (k, v) in Cdag)
    N_op = Dict(k => Cdag[k] * C_op[k] for k in keys(Cdag))
    N_up_op = zeros(Float64, dim, dim)
    N_dn_op = zeros(Float64, dim, dim)
    for i in 1:N_sites
        N_up_op .+= N_op["$(i)u"]
        N_dn_op .+= N_op["$(i)d"]
    end
    return N_up_op, N_dn_op
end

function solve_ed_state_vector(hamil::Matrix{Float64})
    F = eigen(Symmetric(hamil))
    return F.vectors[:, 1]
end

# ==========================================
# 2. 動的近似手法 (HFのみ)
# ==========================================
function solve_hf_orbitals_dynamic(t_x, t_y, U, mu, N_up_count, N_dn_count, N_sites=6)
    T = zeros(Float64, N_sites, N_sites)
    edges = [(1, 2, t_x), (2, 3, t_x), (4, 5, t_x), (5, 6, t_x),
             (1, 4, t_y), (2, 5, t_y), (3, 6, t_y)]
    for (i, j, t) in edges
        T[i, j] = -t
        T[j, i] = -t
    end
    
    n_up, n_dn = zeros(N_sites), zeros(N_sites)
    for i in 1:N_up_count; n_up[i] = 0.8; end
    for i in 1:N_dn_count; n_dn[i] = 0.8; end
    if N_up_count > 0; n_up[1] += 0.1; end
    if N_dn_count > 0; n_dn[1] -= 0.1; end

    ev_up, ev_dn = zeros(N_sites, N_sites), zeros(N_sites, N_sites)
    for iteration in 1:200
        H_up = T + diagm(U .* n_dn) - mu * I
        H_dn = T + diagm(U .* n_up) - mu * I
        F_up, F_dn = eigen(Symmetric(H_up)), eigen(Symmetric(H_dn))
        ev_up, ev_dn = F_up.vectors, F_dn.vectors
        
        new_n_up, new_n_dn = zeros(N_sites), zeros(N_sites)
        for i in 1:N_up_count; new_n_up .+= ev_up[:, i].^2; end
        for i in 1:N_dn_count; new_n_dn .+= ev_dn[:, i].^2; end
            
        if maximum(abs.(new_n_up .- n_up)) < 1e-8; break; end
        n_up = 0.5 .* new_n_up .+ 0.5 .* n_up
        n_dn = 0.5 .* new_n_dn .+ 0.5 .* n_dn
    end
    return ev_up, ev_dn
end

function build_slater_determinant(up_idx, dn_idx, D_up, D_dn, N_sites=6)
    dim = 2^(2 * N_sites)
    vac = zeros(dim); vac[1] = 1.0
    res = vac
    for i in reverse(dn_idx); res = D_dn[i] * res; end
    for i in reverse(up_idx); res = D_up[i] * res; end
    return res / norm(res)
end

function get_hf_state_vector_dynamic(ev_up, ev_dn, N_up_count, N_dn_count, N_sites=6)
    Cdag = get_creation_ops(N_sites)
    D_up = [sum(ev_up[s, k] * Cdag["$(s)u"] for s in 1:N_sites) for k in 1:N_sites]
    D_dn = [sum(ev_dn[s, k] * Cdag["$(s)d"] for s in 1:N_sites) for k in 1:N_sites]
    
    occ_up = collect(1:N_up_count)
    occ_dn = collect(1:N_dn_count)
    
    return build_slater_determinant(occ_up, occ_dn, D_up, D_dn, N_sites)
end

# ==========================================
# 3. 高速化シャドウサンプリング
# ==========================================
function get_measurement_probs(state_vec, u_list, N_sites=6)
    N_qubits = 2 * N_sites
    U_global = ones(ComplexF64, 1, 1)
    for i in N_qubits:-1:1
        U_global = kron(u_list[i], U_global)
    end
    return abs2.(U_global * state_vec)
end

function sample_index(probs, N_sites=6)
    dim = 2^(2 * N_sites)
    r, cp = rand(), 0.0
    for i in 1:dim
        cp += probs[i]
        if r <= cp; return i - 1; end
    end
    return dim - 1
end

function build_snapshot(idx, u_list, N_sites=6)
    N_qubits = 2 * N_sites
    bits = reverse(digits(idx, base=2, pad=N_qubits))
    I2 = ComplexF64[1 0; 0 1]
    rho = ones(ComplexF64, 1, 1)
    for i in N_qubits:-1:1
        u, b = u_list[i], bits[i]
        s_b = (b == 0) ? [1.0, 0.0] : [0.0, 1.0]
        rho = kron(3.0 .* (u' * (s_b * s_b') * u) - I2, rho)
    end
    return rho
end

# ==========================================
# 4. メイン実行部
# ==========================================
function main()
    N_sites = 6
    t_x, t_y = 1.0, 1.0
    
    # 4096次元の密行列計算はメモリを圧迫するため、サンプリング数を低めにしています。
    # 余裕があれば以前の (nu=100, nm=200) まで増やしてください。
    settings_list = [(nu=10, nm=10), (nu=20, nm=20), (nu=30, nm=30)]
    n_repeat = 10
    U_list = [0.1, 4.0]
    
    total_shots_list = [s.nu * s.nm for s in settings_list]
    plots_array = []
    
    N_up_op, N_dn_op = get_particle_number_ops(N_sites)
    
    println("=== Starting 2D (3x2) Dynamic CRM Simulation with HF Prior ===")
    println("※ $N_sites サイト (12量子ビット, 4096次元) の計算を行います。")
    
    for U in U_list
        @printf("\nProcessing U = %.1f ...\n", U)
        mu = 0.0 
        
        hamil = build_hamiltonian(t_x, t_y, U, mu, N_sites)
        vec_ed = solve_ed_state_vector(hamil)
        
        N_up_count = round(Int, real(vec_ed' * N_up_op * vec_ed))
        N_dn_count = round(Int, real(vec_ed' * N_dn_op * vec_ed))
        @printf("  -> ED determined particle sector: Up = %d, Dn = %d\n", N_up_count, N_dn_count)
        
        ev_up, ev_dn = solve_hf_orbitals_dynamic(t_x, t_y, U, mu, N_up_count, N_dn_count, N_sites)
        vec_hf = get_hf_state_vector_dynamic(ev_up, ev_dn, N_up_count, N_dn_count, N_sites)
        
        true_fid = abs(dot(vec_ed, vec_hf))^2
        @printf("  -> Dynamic HF True Fidelity: %.4f\n", true_fid)
        
        m_std, e_std, m_crm, e_crm = Float64[], Float64[], Float64[], Float64[]
        
        for (nu, nm) in settings_list
            tmp_std, tmp_crm = Float64[], Float64[]
            @showprogress 1 "  Simulating (nu=$(lpad(nu,3)), nm=$(lpad(nm,3))) : " for rep in 1:n_repeat
                v_std_nu, v_crm_nu = 0.0, 0.0
                for r in 1:nu
                    u_l = [rand([ComplexF64[1 0; 0 1], [1 1; 1 -1]/sqrt(2), [1 -im; 1 im]/sqrt(2)]) for _ in 1:(2*N_sites)]
                    p_ed = get_measurement_probs(vec_ed, u_l, N_sites)
                    p_hf = get_measurement_probs(vec_hf, u_l, N_sites)
                    
                    s_ed, s_hf = 0.0, 0.0
                    for m in 1:nm
                        s_ed += real(vec_hf' * build_snapshot(sample_index(p_ed, N_sites), u_l, N_sites) * vec_hf)
                        s_hf += real(vec_hf' * build_snapshot(sample_index(p_hf, N_sites), u_l, N_sites) * vec_hf)
                    end
                    v_std_nu += s_ed / nm
                    v_crm_nu += (s_ed - s_hf) / nm
                end
                push!(tmp_std, v_std_nu / nu)
                push!(tmp_crm, (v_crm_nu / nu) + 1.0)
            end
            push!(m_std, mean(tmp_std)); push!(e_std, std(tmp_std))
            push!(m_crm, mean(tmp_crm)); push!(e_crm, std(tmp_crm))
        end
        
        p = plot(total_shots_list, fill(true_fid, length(total_shots_list)), 
                 label="True Fidelity", lw=2, color=:black, linestyle=:dash, xscale=:log10)
        plot!(p, total_shots_list, m_std, yerror=e_std, label="Standard Shadow", marker=:circle, color=:blue, msc=:blue, alpha=0.6)
        plot!(p, total_shots_list, m_crm, yerror=e_crm, label="CRM (HF Prior)", marker=:square, color=:red, msc=:red, alpha=0.6)
        title!(p, "U = $U (HF Fid: $(round(true_fid, digits=3)))")
        xlabel!(p, "Total Shots")
        ylabel!(p, "Estimated Fidelity")
        push!(plots_array, p)
    end
    
    final_plot = plot(plots_array..., layout=(1, 2), size=(900, 400), margin=5Plots.mm)

    println("\nプロットを生成中...")
    timestamp_str = Dates.format(now(), "yyyy-mm-dd_HHMMSS")
    file_name = "crm_hf_2D_3x2_result_$(timestamp_str).png"

    save_full_path = joinpath(desktop_path, file_name)
    savefig(final_plot, save_full_path)
    println("完了しました。画像を確認してください: $file_name")
end

main()

Images will be saved to: /Users/tatsuyamahiroshitaira/Desktop/GraphResults_Hubbard_CRM_HF_2D
=== Starting 2D (3x2) Dynamic CRM Simulation with HF Prior ===
※ 6 サイト (12量子ビット, 4096次元) の計算を行います。

Processing U = 0.1 ...
  -> ED determined particle sector: Up = 3, Dn = 3
  -> Dynamic HF True Fidelity: 0.9996


  Simulating (nu= 10, nm= 10) : 100%|███████████████████| Time: 0:09:40
  Simulating (nu= 20, nm= 20) : 100%|███████████████████| Time: 5:19:33
  Simulating (nu= 30, nm= 30) : 100%|███████████████████| Time: 17:03:17



Processing U = 4.0 ...
  -> ED determined particle sector: Up = 2, Dn = 2
  -> Dynamic HF True Fidelity: 0.4604


  Simulating (nu= 10, nm= 10) : 100%|███████████████████| Time: 0:08:48
  Simulating (nu= 20, nm= 20) : 100%|███████████████████| Time: 0:43:41
  Simulating (nu= 30, nm= 30) : 100%|███████████████████| Time: 5:23:43



プロットを生成中...
完了しました。画像を確認してください: crm_hf_2D_3x2_result_2026-05-15_193420.png


In [3]:
using LinearAlgebra
using Printf
using Statistics
using Plots
using Dates
using ProgressMeter

# --- 0. 保存先の準備 ---
desktop_path = joinpath(homedir(), "Desktop", "GraphResults_Hubbard_CRM_HF_2x2")
mkpath(desktop_path)
println("Images will be saved to: $desktop_path")

# ==========================================
# 1. ハミルトニアンと演算子の構築関数 (汎用化)
# ==========================================
function make_matrix(ops...)
    r = ones(1, 1)
    for op in reverse(ops)
        r = kron(op, r)
    end
    return r
end

function get_creation_ops(N_sites=4)
    cdag = [0.0 0.0; 1.0 0.0]
    I2   = [1.0 0.0; 0.0 1.0]
    F_op = [1.0 0.0; 0.0 -1.0]
    Cdag = Dict{String, Matrix{Float64}}()
    N_qubits = 2 * N_sites
    
    for i in 1:N_sites
        for (s_idx, s_name) in enumerate(["u", "d"])
            q_idx = 2 * (i - 1) + s_idx
            ops = Matrix{Float64}[]
            for j in 1:N_qubits
                if j < q_idx
                    push!(ops, I2)
                elseif j == q_idx
                    push!(ops, cdag)
                else
                    push!(ops, F_op)
                end
            end
            Cdag["$(i)$(s_name)"] = make_matrix(ops...)
        end
    end
    return Cdag
end

function build_hamiltonian(t_x, t_y, U, mu, N_sites=4)
    dim = 2^(2 * N_sites)
    Cdag = get_creation_ops(N_sites)
    C_op = Dict(k => v' for (k, v) in Cdag)
    N_op = Dict(k => Cdag[k] * C_op[k] for k in keys(Cdag))
    hamil = zeros(Float64, dim, dim)
    
    # 2x2 のエッジ定義
    # 1 - 2
    # |   |
    # 3 - 4
    edges = [(1, 2, t_x), (3, 4, t_x),  # 水平 (x方向)
             (1, 3, t_y), (2, 4, t_y)]  # 垂直 (y方向)
             
    for (i, j, t) in edges
        for s in ["u", "d"]
            k1, k2 = "$(i)$s", "$(j)$s"
            hamil .+= -t .* (Cdag[k1] * C_op[k2] .+ Cdag[k2] * C_op[k1])
        end
    end
    
    for i in 1:N_sites
        hamil .+= U .* (N_op["$(i)u"] * N_op["$(i)d"])
        hamil .+= -mu .* N_op["$(i)u"]
        hamil .+= -mu .* N_op["$(i)d"]
    end
    return hamil
end

function get_particle_number_ops(N_sites=4)
    dim = 2^(2 * N_sites)
    Cdag = get_creation_ops(N_sites)
    C_op = Dict(k => v' for (k, v) in Cdag)
    N_op = Dict(k => Cdag[k] * C_op[k] for k in keys(Cdag))
    N_up_op = zeros(Float64, dim, dim)
    N_dn_op = zeros(Float64, dim, dim)
    for i in 1:N_sites
        N_up_op .+= N_op["$(i)u"]
        N_dn_op .+= N_op["$(i)d"]
    end
    return N_up_op, N_dn_op
end

function solve_ed_state_vector(hamil::Matrix{Float64})
    F = eigen(Symmetric(hamil))
    return F.vectors[:, 1]
end

# ==========================================
# 2. 動的近似手法 (HFのみ)
# ==========================================
function solve_hf_orbitals_dynamic(t_x, t_y, U, mu, N_up_count, N_dn_count, N_sites=4)
    T = zeros(Float64, N_sites, N_sites)
    edges = [(1, 2, t_x), (3, 4, t_x), (1, 3, t_y), (2, 4, t_y)]
    for (i, j, t) in edges
        T[i, j] = -t
        T[j, i] = -t
    end
    
    n_up, n_dn = zeros(N_sites), zeros(N_sites)
    for i in 1:N_up_count; n_up[i] = 0.8; end
    for i in 1:N_dn_count; n_dn[i] = 0.8; end
    if N_up_count > 0; n_up[1] += 0.1; end
    if N_dn_count > 0; n_dn[1] -= 0.1; end

    ev_up, ev_dn = zeros(N_sites, N_sites), zeros(N_sites, N_sites)
    for iteration in 1:200
        H_up = T + diagm(U .* n_dn) - mu * I
        H_dn = T + diagm(U .* n_up) - mu * I
        F_up, F_dn = eigen(Symmetric(H_up)), eigen(Symmetric(H_dn))
        ev_up, ev_dn = F_up.vectors, F_dn.vectors
        
        new_n_up, new_n_dn = zeros(N_sites), zeros(N_sites)
        for i in 1:N_up_count; new_n_up .+= ev_up[:, i].^2; end
        for i in 1:N_dn_count; new_n_dn .+= ev_dn[:, i].^2; end
            
        if maximum(abs.(new_n_up .- n_up)) < 1e-8; break; end
        n_up = 0.5 .* new_n_up .+ 0.5 .* n_up
        n_dn = 0.5 .* new_n_dn .+ 0.5 .* n_dn
    end
    return ev_up, ev_dn
end

function build_slater_determinant(up_idx, dn_idx, D_up, D_dn, N_sites=4)
    dim = 2^(2 * N_sites)
    vac = zeros(dim); vac[1] = 1.0
    res = vac
    for i in reverse(dn_idx); res = D_dn[i] * res; end
    for i in reverse(up_idx); res = D_up[i] * res; end
    return res / norm(res)
end

function get_hf_state_vector_dynamic(ev_up, ev_dn, N_up_count, N_dn_count, N_sites=4)
    Cdag = get_creation_ops(N_sites)
    D_up = [sum(ev_up[s, k] * Cdag["$(s)u"] for s in 1:N_sites) for k in 1:N_sites]
    D_dn = [sum(ev_dn[s, k] * Cdag["$(s)d"] for s in 1:N_sites) for k in 1:N_sites]
    
    occ_up = collect(1:N_up_count)
    occ_dn = collect(1:N_dn_count)
    
    return build_slater_determinant(occ_up, occ_dn, D_up, D_dn, N_sites)
end

# ==========================================
# 3. 高速化シャドウサンプリング
# ==========================================
function get_measurement_probs(state_vec, u_list, N_sites=4)
    N_qubits = 2 * N_sites
    U_global = ones(ComplexF64, 1, 1)
    for i in N_qubits:-1:1
        U_global = kron(u_list[i], U_global)
    end
    return abs2.(U_global * state_vec)
end

function sample_index(probs, N_sites=4)
    dim = 2^(2 * N_sites)
    r, cp = rand(), 0.0
    for i in 1:dim
        cp += probs[i]
        if r <= cp; return i - 1; end
    end
    return dim - 1
end

function build_snapshot(idx, u_list, N_sites=4)
    N_qubits = 2 * N_sites
    bits = reverse(digits(idx, base=2, pad=N_qubits))
    I2 = ComplexF64[1 0; 0 1]
    rho = ones(ComplexF64, 1, 1)
    for i in N_qubits:-1:1
        u, b = u_list[i], bits[i]
        s_b = (b == 0) ? [1.0, 0.0] : [0.0, 1.0]
        rho = kron(3.0 .* (u' * (s_b * s_b') * u) - I2, rho)
    end
    return rho
end

# ==========================================
# 4. メイン実行部
# ==========================================
function main()
    N_sites = 4
    t_x, t_y = 1.0, 1.0
    
    # 計算が軽いのでサンプリング数を十分に取ります
    settings_list = [(nu=10, nm=10), (nu=20, nm=50), (nu=50, nm=100), (nu=100, nm=200)]
    n_repeat = 20
    U_list = [0.1, 4.0]
    
    total_shots_list = [s.nu * s.nm for s in settings_list]
    plots_array = []
    
    N_up_op, N_dn_op = get_particle_number_ops(N_sites)
    
    println("=== Starting 2D (2x2) Dynamic CRM Simulation with HF Prior ===")
    println("※ $N_sites サイト (8量子ビット, 256次元) の計算を行います。")
    
    for U in U_list
        @printf("\nProcessing U = %.1f ...\n", U)
        mu = U/2.0  # 半充填に設定
        
        hamil = build_hamiltonian(t_x, t_y, U, mu, N_sites)
        vec_ed = solve_ed_state_vector(hamil)
        
        N_up_count = round(Int, real(vec_ed' * N_up_op * vec_ed))
        N_dn_count = round(Int, real(vec_ed' * N_dn_op * vec_ed))
        @printf("  -> ED determined particle sector: Up = %d, Dn = %d\n", N_up_count, N_dn_count)
        
        ev_up, ev_dn = solve_hf_orbitals_dynamic(t_x, t_y, U, mu, N_up_count, N_dn_count, N_sites)
        vec_hf = get_hf_state_vector_dynamic(ev_up, ev_dn, N_up_count, N_dn_count, N_sites)
        
        true_fid = abs(dot(vec_ed, vec_hf))^2
        @printf("  -> Dynamic HF True Fidelity: %.4f\n", true_fid)
        
        m_std, e_std, m_crm, e_crm = Float64[], Float64[], Float64[], Float64[]
        
        for (nu, nm) in settings_list
            tmp_std, tmp_crm = Float64[], Float64[]
            @showprogress 1 "  Simulating (nu=$(lpad(nu,3)), nm=$(lpad(nm,3))) : " for rep in 1:n_repeat
                v_std_nu, v_crm_nu = 0.0, 0.0
                for r in 1:nu
                    u_l = [rand([ComplexF64[1 0; 0 1], [1 1; 1 -1]/sqrt(2), [1 -im; 1 im]/sqrt(2)]) for _ in 1:(2*N_sites)]
                    p_ed = get_measurement_probs(vec_ed, u_l, N_sites)
                    p_hf = get_measurement_probs(vec_hf, u_l, N_sites)
                    
                    s_ed, s_hf = 0.0, 0.0
                    for m in 1:nm
                        s_ed += real(vec_hf' * build_snapshot(sample_index(p_ed, N_sites), u_l, N_sites) * vec_hf)
                        s_hf += real(vec_hf' * build_snapshot(sample_index(p_hf, N_sites), u_l, N_sites) * vec_hf)
                    end
                    v_std_nu += s_ed / nm
                    v_crm_nu += (s_ed - s_hf) / nm
                end
                push!(tmp_std, v_std_nu / nu)
                push!(tmp_crm, (v_crm_nu / nu) + 1.0)
            end
            push!(m_std, mean(tmp_std)); push!(e_std, std(tmp_std))
            push!(m_crm, mean(tmp_crm)); push!(e_crm, std(tmp_crm))
        end
        
        p = plot(total_shots_list, fill(true_fid, length(total_shots_list)), 
                 label="True Fidelity", lw=2, color=:black, linestyle=:dash, xscale=:log10)
        plot!(p, total_shots_list, m_std, yerror=e_std, label="Standard Shadow", marker=:circle, color=:blue, msc=:blue, alpha=0.6)
        plot!(p, total_shots_list, m_crm, yerror=e_crm, label="CRM (HF Prior)", marker=:square, color=:red, msc=:red, alpha=0.6)
        title!(p, "U = $U (HF Fid: $(round(true_fid, digits=3)))")
        xlabel!(p, "Total Shots")
        ylabel!(p, "Estimated Fidelity")
        push!(plots_array, p)
    end
    
    final_plot = plot(plots_array..., layout=(1, 2), size=(900, 400), margin=5Plots.mm)

    println("\nプロットを生成中...")
    timestamp_str = Dates.format(now(), "yyyy-mm-dd_HHMMSS")
    file_name = "crm_hf_2D_2x2_result_$(timestamp_str).png"

    save_full_path = joinpath(desktop_path, file_name)
    savefig(final_plot, save_full_path)
    println("完了しました。画像を確認してください: $file_name")
end

main()

Images will be saved to: /Users/tatsuyamahiroshitaira/Desktop/GraphResults_Hubbard_CRM_HF_2x2
=== Starting 2D (2x2) Dynamic CRM Simulation with HF Prior ===
※ 4 サイト (8量子ビット, 256次元) の計算を行います。

Processing U = 0.1 ...
  -> ED determined particle sector: Up = 2, Dn = 2
  -> Dynamic HF True Fidelity: 0.4999


  Simulating (nu= 10, nm= 10) : 100%|███████████████████| Time: 0:00:04
  Simulating (nu= 20, nm= 50) : 100%|███████████████████| Time: 0:00:29
  Simulating (nu= 50, nm=100) : 100%|███████████████████| Time: 0:02:17
  Simulating (nu=100, nm=200) : 100%|███████████████████| Time: 0:09:42



Processing U = 4.0 ...
  -> ED determined particle sector: Up = 2, Dn = 2
  -> Dynamic HF True Fidelity: 0.4292


  Simulating (nu= 10, nm= 10) : 100%|███████████████████| Time: 0:00:03
  Simulating (nu= 20, nm= 50) : 100%|███████████████████| Time: 0:00:29
  Simulating (nu= 50, nm=100) : 100%|███████████████████| Time: 0:02:24
  Simulating (nu=100, nm=200) : 100%|███████████████████| Time: 0:09:37



プロットを生成中...
完了しました。画像を確認してください: crm_hf_2D_2x2_result_2026-05-20_093715.png


In [4]:
using LinearAlgebra
using Printf
using Statistics
using Plots
using Dates
using ProgressMeter

# --- 0. 保存先の準備 ---
desktop_path = joinpath(homedir(), "Desktop", "GraphResults_Hubbard_CRM_CISD_2x2")
mkpath(desktop_path)
println("Images will be saved to: $desktop_path")

# ==========================================
# 1. ハミルトニアンと演算子の構築関数 (汎用化)
# ==========================================
function make_matrix(ops...)
    r = ones(1, 1)
    for op in reverse(ops)
        r = kron(op, r)
    end
    return r
end

function get_creation_ops(N_sites=4)
    cdag = [0.0 0.0; 1.0 0.0]
    I2   = [1.0 0.0; 0.0 1.0]
    F_op = [1.0 0.0; 0.0 -1.0]
    Cdag = Dict{String, Matrix{Float64}}()
    N_qubits = 2 * N_sites
    
    for i in 1:N_sites
        for (s_idx, s_name) in enumerate(["u", "d"])
            q_idx = 2 * (i - 1) + s_idx
            ops = Matrix{Float64}[]
            for j in 1:N_qubits
                if j < q_idx
                    push!(ops, I2)
                elseif j == q_idx
                    push!(ops, cdag)
                else
                    push!(ops, F_op)
                end
            end
            Cdag["$(i)$(s_name)"] = make_matrix(ops...)
        end
    end
    return Cdag
end

function build_hamiltonian(t_x, t_y, U, mu, N_sites=4)
    dim = 2^(2 * N_sites)
    Cdag = get_creation_ops(N_sites)
    C_op = Dict(k => v' for (k, v) in Cdag)
    N_op = Dict(k => Cdag[k] * C_op[k] for k in keys(Cdag))
    hamil = zeros(Float64, dim, dim)
    
    # 2x2 のエッジ定義
    # 1 - 2
    # |   |
    # 3 - 4
    edges = [(1, 2, t_x), (3, 4, t_x),  # 水平 (x方向)
             (1, 3, t_y), (2, 4, t_y)]  # 垂直 (y方向)
             
    for (i, j, t) in edges
        for s in ["u", "d"]
            k1, k2 = "$(i)$s", "$(j)$s"
            hamil .+= -t .* (Cdag[k1] * C_op[k2] .+ Cdag[k2] * C_op[k1])
        end
    end
    
    for i in 1:N_sites
        hamil .+= U .* (N_op["$(i)u"] * N_op["$(i)d"])
        hamil .+= -mu .* N_op["$(i)u"]
        hamil .+= -mu .* N_op["$(i)d"]
    end
    return hamil
end

function get_particle_number_ops(N_sites=4)
    dim = 2^(2 * N_sites)
    Cdag = get_creation_ops(N_sites)
    C_op = Dict(k => v' for (k, v) in Cdag)
    N_op = Dict(k => Cdag[k] * C_op[k] for k in keys(Cdag))
    N_up_op = zeros(Float64, dim, dim)
    N_dn_op = zeros(Float64, dim, dim)
    for i in 1:N_sites
        N_up_op .+= N_op["$(i)u"]
        N_dn_op .+= N_op["$(i)d"]
    end
    return N_up_op, N_dn_op
end

function solve_ed_state_vector(hamil::Matrix{Float64})
    F = eigen(Symmetric(hamil))
    return F.vectors[:, 1]
end

# ==========================================
# 2. 事前状態の構築 (HF軌道最適化 + CISD)
# ==========================================
function solve_hf_orbitals_dynamic(t_x, t_y, U, mu, N_up_count, N_dn_count, N_sites=4)
    T = zeros(Float64, N_sites, N_sites)
    edges = [(1, 2, t_x), (3, 4, t_x), (1, 3, t_y), (2, 4, t_y)]
    for (i, j, t) in edges
        T[i, j] = -t
        T[j, i] = -t
    end
    
    n_up, n_dn = zeros(N_sites), zeros(N_sites)
    for i in 1:N_up_count; n_up[i] = 0.8; end
    for i in 1:N_dn_count; n_dn[i] = 0.8; end
    if N_up_count > 0; n_up[1] += 0.1; end
    if N_dn_count > 0; n_dn[1] -= 0.1; end

    ev_up, ev_dn = zeros(N_sites, N_sites), zeros(N_sites, N_sites)
    for iteration in 1:200
        H_up = T + diagm(U .* n_dn) - mu * I
        H_dn = T + diagm(U .* n_up) - mu * I
        F_up, F_dn = eigen(Symmetric(H_up)), eigen(Symmetric(H_dn))
        ev_up, ev_dn = F_up.vectors, F_dn.vectors
        
        new_n_up, new_n_dn = zeros(N_sites), zeros(N_sites)
        for i in 1:N_up_count; new_n_up .+= ev_up[:, i].^2; end
        for i in 1:N_dn_count; new_n_dn .+= ev_dn[:, i].^2; end
            
        if maximum(abs.(new_n_up .- n_up)) < 1e-8; break; end
        n_up = 0.5 .* new_n_up .+ 0.5 .* n_up
        n_dn = 0.5 .* new_n_dn .+ 0.5 .* n_dn
    end
    return ev_up, ev_dn
end

function build_slater_determinant(up_idx, dn_idx, D_up, D_dn, N_sites=4)
    dim = 2^(2 * N_sites)
    vac = zeros(dim); vac[1] = 1.0
    res = vac
    for i in reverse(dn_idx); res = D_dn[i] * res; end
    for i in reverse(up_idx); res = D_up[i] * res; end
    return res / norm(res)
end

# 組み合わせを生成するユーティリティ関数
function simple_combinations(arr, k)
    k == 0 && return [Int[]]
    k > length(arr) && return []
    res = Vector{Int}[]
    for i in 1:(length(arr) - k + 1)
        for rest in simple_combinations(arr[i+1:end], k-1)
            push!(res, vcat(arr[i], rest))
        end
    end
    return res
end

# ★ 変更点: CISD部分空間の構築と対角化を行う関数
function get_cisd_state_vector(hamil, ev_up, ev_dn, N_up_count, N_dn_count, N_sites=4)
    Cdag = get_creation_ops(N_sites)
    D_up = [sum(ev_up[s, k] * Cdag["$(s)u"] for s in 1:N_sites) for k in 1:N_sites]
    D_dn = [sum(ev_dn[s, k] * Cdag["$(s)d"] for s in 1:N_sites) for k in 1:N_sites]
    
    hf_up = collect(1:N_up_count)
    hf_dn = collect(1:N_dn_count)
    
    # 可能なすべての電子配置を生成
    all_up_configs = simple_combinations(collect(1:N_sites), N_up_count)
    all_dn_configs = simple_combinations(collect(1:N_sites), N_dn_count)
    
    basis_vecs = Vector{Float64}[]
    
    # HF状態から2電子励起以下の状態のみを抽出して基底に追加
    for up_conf in all_up_configs
        for dn_conf in all_dn_configs
            exc_up = length(setdiff(up_conf, hf_up))
            exc_dn = length(setdiff(dn_conf, hf_dn))
            if exc_up + exc_dn <= 2
                vec = build_slater_determinant(up_conf, dn_conf, D_up, D_dn, N_sites)
                push!(basis_vecs, vec)
            end
        end
    end
    
    num_basis = length(basis_vecs)
    H_sub = zeros(Float64, num_basis, num_basis)
    
    # CISD部分空間でハミルトニアンの行列要素を計算
    for i in 1:num_basis
        H_vec = hamil * basis_vecs[i]
        for j in i:num_basis
            val = real(dot(basis_vecs[j], H_vec))
            H_sub[i, j] = val
            H_sub[j, i] = val
        end
    end
    
    # 部分空間の対角化
    F = eigen(Symmetric(H_sub))
    ground_coeff = F.vectors[:, 1]
    
    # 元の巨大空間 (256次元) のベクトルに復元
    cisd_vec = zeros(Float64, 2^(2 * N_sites))
    for i in 1:num_basis
        cisd_vec .+= ground_coeff[i] .* basis_vecs[i]
    end
    return cisd_vec / norm(cisd_vec)
end

# ==========================================
# 3. 高速化シャドウサンプリング
# ==========================================
function get_measurement_probs(state_vec, u_list, N_sites=4)
    N_qubits = 2 * N_sites
    U_global = ones(ComplexF64, 1, 1)
    for i in N_qubits:-1:1
        U_global = kron(u_list[i], U_global)
    end
    return abs2.(U_global * state_vec)
end

function sample_index(probs, N_sites=4)
    dim = 2^(2 * N_sites)
    r, cp = rand(), 0.0
    for i in 1:dim
        cp += probs[i]
        if r <= cp; return i - 1; end
    end
    return dim - 1
end

function build_snapshot(idx, u_list, N_sites=4)
    N_qubits = 2 * N_sites
    bits = reverse(digits(idx, base=2, pad=N_qubits))
    I2 = ComplexF64[1 0; 0 1]
    rho = ones(ComplexF64, 1, 1)
    for i in N_qubits:-1:1
        u, b = u_list[i], bits[i]
        s_b = (b == 0) ? [1.0, 0.0] : [0.0, 1.0]
        rho = kron(3.0 .* (u' * (s_b * s_b') * u) - I2, rho)
    end
    return rho
end

# ==========================================
# 4. メイン実行部
# ==========================================
function main()
    N_sites = 4
    t_x, t_y = 1.0, 1.0
    
    settings_list = [(nu=10, nm=10), (nu=20, nm=50), (nu=50, nm=100), (nu=100, nm=200)]
    n_repeat = 20
    U_list = [0.1, 4.0]
    
    total_shots_list = [s.nu * s.nm for s in settings_list]
    plots_array = []
    
    N_up_op, N_dn_op = get_particle_number_ops(N_sites)
    
    println("=== Starting 2D (2x2) Dynamic CRM Simulation with CISD Prior ===")
    println("※ $N_sites サイト (8量子ビット, 256次元) の計算を行います。")
    
    for U in U_list
        @printf("\nProcessing U = %.1f ...\n", U)
        mu = U/2.0  # 半充填に設定
        
        hamil = build_hamiltonian(t_x, t_y, U, mu, N_sites)
        vec_ed = solve_ed_state_vector(hamil)
        
        N_up_count = round(Int, real(vec_ed' * N_up_op * vec_ed))
        N_dn_count = round(Int, real(vec_ed' * N_dn_op * vec_ed))
        @printf("  -> ED determined particle sector: Up = %d, Dn = %d\n", N_up_count, N_dn_count)
        
        # HF軌道を最適化し、その軌道を使ってCISD状態を構築
        ev_up, ev_dn = solve_hf_orbitals_dynamic(t_x, t_y, U, mu, N_up_count, N_dn_count, N_sites)
        vec_cisd = get_cisd_state_vector(hamil, ev_up, ev_dn, N_up_count, N_dn_count, N_sites)
        
        true_fid = abs(dot(vec_ed, vec_cisd))^2
        @printf("  -> Dynamic CISD True Fidelity: %.4f\n", true_fid)
        
        m_std, e_std, m_crm, e_crm = Float64[], Float64[], Float64[], Float64[]
        
        for (nu, nm) in settings_list
            tmp_std, tmp_crm = Float64[], Float64[]
            @showprogress 1 "  Simulating (nu=$(lpad(nu,3)), nm=$(lpad(nm,3))) : " for rep in 1:n_repeat
                v_std_nu, v_crm_nu = 0.0, 0.0
                for r in 1:nu
                    u_l = [rand([ComplexF64[1 0; 0 1], [1 1; 1 -1]/sqrt(2), [1 -im; 1 im]/sqrt(2)]) for _ in 1:(2*N_sites)]
                    p_ed = get_measurement_probs(vec_ed, u_l, N_sites)
                    p_cisd = get_measurement_probs(vec_cisd, u_l, N_sites)
                    
                    s_ed, s_cisd = 0.0, 0.0
                    for m in 1:nm
                        s_ed += real(vec_cisd' * build_snapshot(sample_index(p_ed, N_sites), u_l, N_sites) * vec_cisd)
                        s_cisd += real(vec_cisd' * build_snapshot(sample_index(p_cisd, N_sites), u_l, N_sites) * vec_cisd)
                    end
                    v_std_nu += s_ed / nm
                    v_crm_nu += (s_ed - s_cisd) / nm
                end
                push!(tmp_std, v_std_nu / nu)
                push!(tmp_crm, (v_crm_nu / nu) + 1.0)
            end
            push!(m_std, mean(tmp_std)); push!(e_std, std(tmp_std))
            push!(m_crm, mean(tmp_crm)); push!(e_crm, std(tmp_crm))
        end
        
        p = plot(total_shots_list, fill(true_fid, length(total_shots_list)), 
                 label="True Fidelity", lw=2, color=:black, linestyle=:dash, xscale=:log10)
        plot!(p, total_shots_list, m_std, yerror=e_std, label="Standard Shadow", marker=:circle, color=:blue, msc=:blue, alpha=0.6)
        plot!(p, total_shots_list, m_crm, yerror=e_crm, label="CRM (CISD Prior)", marker=:square, color=:red, msc=:red, alpha=0.6)
        title!(p, "U = $U (CISD Fid: $(round(true_fid, digits=3)))")
        xlabel!(p, "Total Shots")
        ylabel!(p, "Estimated Fidelity")
        push!(plots_array, p)
    end
    
    final_plot = plot(plots_array..., layout=(1, 2), size=(900, 400), margin=5Plots.mm)

    println("\nプロットを生成中...")
    timestamp_str = Dates.format(now(), "yyyy-mm-dd_HHMMSS")
    file_name = "crm_cisd_2D_2x2_result_$(timestamp_str).png"

    save_full_path = joinpath(desktop_path, file_name)
    savefig(final_plot, save_full_path)
    println("完了しました。画像を確認してください: $file_name")
end

main()

Images will be saved to: /Users/tatsuyamahiroshitaira/Desktop/GraphResults_Hubbard_CRM_CISD_2x2
=== Starting 2D (2x2) Dynamic CRM Simulation with CISD Prior ===
※ 4 サイト (8量子ビット, 256次元) の計算を行います。

Processing U = 0.1 ...
  -> ED determined particle sector: Up = 2, Dn = 2
  -> Dynamic CISD True Fidelity: 0.7121


  Simulating (nu= 10, nm= 10) : 100%|███████████████████| Time: 0:00:05
  Simulating (nu= 20, nm= 50) : 100%|███████████████████| Time: 0:00:40
  Simulating (nu= 50, nm=100) : 100%|███████████████████| Time: 0:02:32
  Simulating (nu=100, nm=200) : 100%|███████████████████| Time: 0:09:48



Processing U = 4.0 ...
  -> ED determined particle sector: Up = 2, Dn = 2
  -> Dynamic CISD True Fidelity: 0.6723


  Simulating (nu= 10, nm= 10) : 100%|███████████████████| Time: 0:00:03
  Simulating (nu= 20, nm= 50) : 100%|███████████████████| Time: 0:00:33
  Simulating (nu= 50, nm=100) : 100%|███████████████████| Time: 0:02:25
  Simulating (nu=100, nm=200) : 100%|███████████████████| Time: 0:17:34



プロットを生成中...
完了しました。画像を確認してください: crm_cisd_2D_2x2_result_2026-05-20_103638.png
